# Step 4 Dataset Collection Notebook

This notebook duplicates the Step 4 Python collection logic in a self-contained format for Google Colab. It downloads recent SEC 10-K filings, extracts the MD&A section, splits the text into filtered sentences, and saves the raw dataset to Google Drive at `Team 8 - Capstone Project/Step 4 - Dataset Collection/data/raw_dataset.csv`.

Run the notebook from top to bottom. The collection can take a while because SEC requests are rate limited.

## 1. Install Dependencies, Mount Drive, and Configure Paths

This section installs notebook dependencies, imports the same libraries used by the Step 4 scripts, mounts Google Drive, downloads the NLTK tokenizer, and points outputs to the shared Step 4 folder in Drive.

In [ ]:
# 1. Install Dependencies
# Installs required libraries in a fresh Colab runtime.
%pip install -q beautifulsoup4 lxml nltk pandas requests tqdm

# 2. Imports
import csv
import html as html_module
import os
import re
import time
import unicodedata
import warnings
from collections import Counter
from pathlib import Path
from urllib.parse import parse_qs, unquote, urljoin, urlsplit

from bs4 import BeautifulSoup
import nltk
from nltk.tokenize import sent_tokenize
import pandas as pd
import requests
from tqdm.auto import tqdm

# Silence warnings for cleaner notebook output.
warnings.filterwarnings("ignore")

# 3. Mount Google Drive
# Dynamic import avoids local IDE warnings when google.colab is unavailable outside Colab.
try:
    from importlib import import_module

    drive = import_module("google.colab").drive
except ModuleNotFoundError as exc:
    raise RuntimeError("Run this notebook in Google Colab so Google Drive can be mounted.") from exc

drive.mount("/content/drive")

# 4. Download NLTK Tokenizer
# Required for sentence splitting with sent_tokenize.
nltk.download("punkt")
nltk.download("punkt_tab")

# 5. Set File Paths
# This points to your Step 4 folder in Google Drive.
# Make sure this exactly matches your folder structure:
# MyDrive > Team 8 - Capstone Project > Step 4 - Dataset Collection
DRIVE_BASE = Path("/content/drive/MyDrive/Team 8 - Capstone Project/Step 4 - Dataset Collection")

# Create a data subfolder if it does not already exist.
DATA_DIR = DRIVE_BASE / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Final dataset output path.
OUTPUT_FILE = DATA_DIR / "raw_dataset.csv"

# Confirm where files will be saved.
print(f"Saving dataset to: {OUTPUT_FILE}")

# 6. Collection Parameters
# These values match your Step 4 Python script exactly.
TARGET_SENTENCES = 16500
MAX_SENTENCES_PER_COMPANY = 200
FILINGS_PER_COMPANY = 5
SLEEP_SECONDS = 0.15

# 7. SEC Request Headers
# Required by SEC EDGAR and must include an identity.
HEADERS = {
    "User-Agent": "Lucy Moore lmoore36@unc.edu",
    "Accept-Encoding": "gzip, deflate",
}

# 8. Ticker List (Sampling)
# Diverse set of companies across industries.
TICKERS = [
    # Tech
    "AAPL", "MSFT", "GOOGL", "META", "NVDA", "INTC", "IBM", "ORCL", "CSCO", "ADBE",
    # Healthcare / Pharma
    "JNJ", "PFE", "MRK", "ABT", "BMY", "AMGN", "GILD", "MDT", "UNH", "CVS",
    # Finance
    "JPM", "BAC", "WFC", "GS", "MS", "C", "AXP", "BLK", "COF", "USB",
    # Consumer / Retail
    "WMT", "AMZN", "TGT", "COST", "HD", "LOW", "NKE", "SBUX", "MCD", "YUM",
    # Energy
    "XOM", "CVX", "COP", "SLB", "PSX", "VLO", "MPC", "OXY", "HES", "DVN",
    # Industrials
    "GE", "HON", "MMM", "CAT", "DE", "BA", "LMT", "RTX", "UPS", "FDX",
    # Telecom / Media
    "T", "VZ", "CMCSA", "DIS", "NFLX", "PARA", "WBD", "FOXA", "DISH", "LUMN",
    # Materials / Misc
    "DD", "DOW", "LIN", "APD", "NEM", "FCX", "VMC", "MLM", "PKG", "IP",
]

# 9. Output Schema
# Columns for your dataset. These must stay consistent across steps.
CSV_COLUMNS = [
    "sentence_id",
    "sentence",
    "ticker",
    "cik",
    "filing_date",
    "filing_id",
]

# 10. Final Sanity Check
# Quick test to confirm writing to Drive works before running full collection.
test_df = pd.DataFrame({"check": [1, 2, 3]})
test_df.to_csv(DATA_DIR / "test_file.csv", index=False)

print("Test file written successfully. Check your Google Drive folder.")

## 2. SEC Request Setup

These helper functions make SEC requests with the configured headers and pause after every request to stay within EDGAR rate guidance.

In [ ]:
def request_sec(url, headers, params=None, timeout=10, sleep_seconds=0.15):
    """Make one SEC request and pause so we stay inside EDGAR rate guidance."""
    response = requests.get(url, params=params, headers=headers, timeout=timeout)
    time.sleep(sleep_seconds)
    response.raise_for_status()
    return response


def clean_doc_url(url):
    """Strip the iXBRL viewer wrapper SEC adds to some filing document URLs."""
    if not url:
        return url

    parsed = urlsplit(url)
    if parsed.path == "/ix":
        doc = parse_qs(parsed.query).get("doc", [""])[0]
        if doc:
            return urljoin("https://www.sec.gov", unquote(doc))

    return url

## 3. Filing Retrieval

These functions look up each company's CIK, find recent 10-K filings, build the filing document URL, and download the raw filing HTML.

In [ ]:
def get_cik(ticker, headers, sleep_seconds=0.15):
    """Look up EDGAR's internal company ID for a ticker symbol."""
    cik = get_cik_from_ticker_file(ticker, headers, sleep_seconds=sleep_seconds)
    if cik:
        return cik

    return get_cik_from_company_search(ticker, headers, sleep_seconds=sleep_seconds)


def get_cik_from_ticker_file(ticker, headers, sleep_seconds=0.15):
    """Use SEC's official ticker/CIK/company-name JSON mapping."""
    url = "https://www.sec.gov/files/company_tickers.json"

    try:
        response = request_sec(url, headers, sleep_seconds=sleep_seconds)
        for company in response.json().values():
            if company.get("ticker", "").upper() == ticker.upper():
                return str(company["cik_str"]).zfill(10)
    except Exception as e:
        print(f"  Could not get CIK from ticker file for {ticker}: {e}")

    return None


def get_cik_from_company_search(ticker, headers, sleep_seconds=0.15):
    """Fallback CIK lookup using the older company search page."""
    url = "https://www.sec.gov/cgi-bin/browse-edgar"
    params = {
        "CIK": ticker,
        "type": "10-K",
        "action": "getcompany",
        "output": "atom",
    }

    try:
        response = request_sec(url, headers, params=params, sleep_seconds=sleep_seconds)
        match = re.search(r"CIK=(\d+)", response.url + response.text)
        if match:
            return match.group(1).zfill(10)
    except Exception as e:
        print(f"  Could not get CIK for {ticker}: {e}")

    return None


def get_10k_filings(cik, headers, max_filings=5, sleep_seconds=0.15):
    """Return recent 10-K filing accession numbers and dates for one CIK."""
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    try:
        response = request_sec(url, headers, sleep_seconds=sleep_seconds)
        data = response.json()
        filings = data.get("filings", {}).get("recent", {})
        forms = filings.get("form", [])
        accessions = filings.get("accessionNumber", [])
        dates = filings.get("filingDate", [])
        primary_documents = filings.get("primaryDocument", [])

        results = []
        for form, accession, date, primary_document in zip(
            forms,
            accessions,
            dates,
            primary_documents,
        ):
            if form == "10-K":
                results.append({
                    "accession": accession,
                    "date": date,
                    "primary_document": primary_document,
                })
            if len(results) >= max_filings:
                break

        return results
    except Exception as e:
        print(f"  Could not get filings for CIK {cik}: {e}")
        return []


def get_filing_document_url(
    cik,
    accession_number,
    headers,
    primary_document=None,
    sleep_seconds=0.15,
):
    """Build the raw 10-K document URL from SEC's primaryDocument metadata."""
    del headers, sleep_seconds

    if not primary_document:
        print(f"  Missing primary document for {accession_number}")
        return None

    acc_clean = accession_number.replace("-", "")
    base = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_clean}/"
    return clean_doc_url(urljoin(base, primary_document))


def download_filing_html(doc_url, headers, sleep_seconds=0.15):
    """Download the raw filing HTML after normalizing any iXBRL viewer URL."""
    response = request_sec(
        clean_doc_url(doc_url),
        headers,
        timeout=30,
        sleep_seconds=sleep_seconds,
    )
    return response.text

## 4. MD&A Extraction

This section removes filing markup and uses Item 7 / Item 7A / Item 8 text patterns to isolate the Management's Discussion and Analysis section.

In [ ]:
def ensure_sentence_tokenizer():
    """Download NLTK sentence tokenizer data if it is not already present."""
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)


def extract_mda_from_html(html):
    """Extract the MD&A / Item 7 section from one 10-K HTML document."""
    soup = BeautifulSoup(html, "lxml")

    # Remove tables and non-content tags before searching the text.
    for tag in soup.find_all(["table", "script", "style", "ix:header"]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    text = html_module.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = re.sub(r"\s+", " ", text)

    start_patterns = [
        r"item\s*7[\.\:\-\u2013]?\s*management'?s?\s+discussion\s+and\s+analysis\s+of\s+financial\s+condition\s+and\s+results\s+of\s+operations",
        r"item\s*7[\.\:\-\u2013]?\s*management'?s?\s+discussion\s+and\s+analysis",
        r"management'?s?\s+discussion\s+and\s+analysis",
    ]
    end_patterns = [
        r"item\s*7a[\.\:\-\u2013]?\s*quantitative\s+and\s+qualitative\s+disclosures",
        r"item\s*8[\.\:\-\u2013]?\s*financial\s+statements",
    ]

    best_section = ""
    for start_pattern in start_patterns:
        for start_match in re.finditer(start_pattern, text, re.IGNORECASE):
            start_pos = start_match.start()
            search_from = start_match.end() + 100
            end_pos = min(start_pos + 150000, len(text))

            for end_pattern in end_patterns:
                end_match = re.search(end_pattern, text[search_from:], re.IGNORECASE)
                if end_match:
                    candidate_end = search_from + end_match.start()
                    if candidate_end < end_pos:
                        end_pos = candidate_end

            candidate = text[start_pos:end_pos].strip()

            # Table-of-contents hits usually run only a few words before Item 7A/8.
            if len(candidate) > len(best_section):
                best_section = candidate
            if len(candidate) > 3000:
                return candidate

    return best_section if len(best_section) > 500 else None

## 5. Sentence Splitting

This section turns MD&A text into sentence-level examples and filters out very short, very long, mostly numeric, and obvious boilerplate sentences.

In [ ]:
def split_into_sentences(text):
    """Split MD&A text into clean sentence-level examples for labeling."""
    ensure_sentence_tokenizer()

    sentences = sent_tokenize(text)
    clean = []

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) < 40:
            continue
        if len(sentence) > 600:
            continue

        letters = sum(char.isalpha() for char in sentence)
        if letters / max(len(sentence), 1) < 0.45:
            continue

        skip_phrases = [
            "item 7",
            "item 8",
            "management's discussion",
            "forward-looking statements",
            "table of contents",
            "see notes to consolidated",
            "incorporated by reference",
        ]
        if any(phrase in sentence.lower() for phrase in skip_phrases):
            continue

        clean.append(sentence)

    return clean

## 6. Dataset Assembly

This section loops through tickers and filings, extracts sentences, writes progress after each filing, and resumes if the Google Drive output file already exists.

In [ ]:
def read_saved_dataset(output_file):
    """Read the generated CSV and fail clearly if an existing file is malformed."""
    try:
        df = pd.read_csv(output_file, dtype=str, encoding="utf-8-sig", keep_default_na=False)
    except pd.errors.ParserError as exc:
        raise ValueError(
            f"{output_file} is malformed and cannot be resumed. "
            "Delete or rename the existing raw_dataset.csv in Google Drive, then rerun the notebook."
        ) from exc

    missing_columns = sorted(set(CSV_COLUMNS) - set(df.columns))
    if missing_columns:
        raise ValueError(
            f"{output_file} is missing required columns: {', '.join(missing_columns)}. "
            "Delete or rename the existing raw_dataset.csv in Google Drive, then rerun the notebook."
        )

    return df[CSV_COLUMNS].copy()


def load_existing_progress(output_file):
    """Return already-processed filing IDs and the next sentence number."""
    if not os.path.exists(output_file):
        return set(), 0, Counter()

    rows = read_saved_dataset(output_file)
    existing_ids = set(rows["filing_id"])
    company_counts = Counter(rows["ticker"])
    return existing_ids, len(rows), company_counts


def collect_sentences_for_filing(ticker, cik, filing):
    """Download one filing and return filtered MD&A sentences from it."""
    accession = filing["accession"]
    filing_date = filing["date"]

    print(f"  Processing {accession} ({filing_date})...")

    doc_url = get_filing_document_url(
        cik,
        accession,
        HEADERS,
        primary_document=filing.get("primary_document"),
        sleep_seconds=SLEEP_SECONDS,
    )
    if not doc_url:
        print("    Could not find document URL")
        return []

    print(f"    Document: {doc_url}")

    try:
        html = download_filing_html(doc_url, HEADERS, sleep_seconds=SLEEP_SECONDS)
    except Exception as e:
        print(f"    Could not download filing document: {e}")
        return []

    mda_text = extract_mda_from_html(html)
    if not mda_text:
        print("    Could not extract MD&A section")
        return []

    sentences = split_into_sentences(mda_text)
    print(f"    Extracted {len(sentences)} sentences")
    return sentences


def write_sentences(writer, sentences, sentence_counter, ticker, cik, filing):
    """Write extracted sentences to CSV and return the next sentence counter."""
    for sentence in sentences:
        writer.writerow({
            "sentence_id": f"s{sentence_counter:06d}",
            "sentence": sentence,
            "ticker": ticker,
            "cik": cik,
            "filing_date": filing["date"],
            "filing_id": filing["accession"],
        })
        sentence_counter += 1

    return sentence_counter


def collect_dataset():
    """Run the full Step 4 SEC collection pipeline and return the saved dataset."""
    print("=" * 60)
    print("SEC EDGAR 10-K MD&A Data Collection")
    print("=" * 60)

    os.makedirs(DATA_DIR, exist_ok=True)

    existing_ids, sentence_counter, company_counts = load_existing_progress(OUTPUT_FILE)
    if existing_ids:
        print(f"Resuming - found {sentence_counter} sentences already collected.")

    mode = "a" if existing_ids else "w"
    output_encoding = "utf-8" if existing_ids else "utf-8-sig"
    with open(OUTPUT_FILE, mode, newline="", encoding=output_encoding) as csvfile:
        writer = csv.DictWriter(
            csvfile,
            fieldnames=CSV_COLUMNS,
            quoting=csv.QUOTE_ALL,
            lineterminator="\n",
        )
        if not existing_ids:
            writer.writeheader()

        print(f"\nTarget: {TARGET_SENTENCES} sentences")
        print(f"Companies to process: {len(TICKERS)}")
        print(f"Filings per company: {FILINGS_PER_COMPANY}")
        print("-" * 60)

        for ticker in tqdm(TICKERS, desc="Companies"):
            if sentence_counter >= TARGET_SENTENCES:
                print(f"\nReached target of {TARGET_SENTENCES} sentences. Stopping.")
                break
            if company_counts[ticker] >= MAX_SENTENCES_PER_COMPANY:
                print(f"\n[{ticker}] Skipping - already has {company_counts[ticker]} sentences.")
                continue

            print(f"\n[{ticker}] Looking up CIK...")
            cik = get_cik(ticker, HEADERS, sleep_seconds=SLEEP_SECONDS)
            if not cik:
                print(f"  Skipping {ticker} - could not find CIK")
                continue

            print(f"  CIK: {cik}")
            filings = get_10k_filings(
                cik,
                HEADERS,
                max_filings=FILINGS_PER_COMPANY,
                sleep_seconds=SLEEP_SECONDS,
            )
            print(f"  Found {len(filings)} 10-K filings")

            for filing in filings:
                accession = filing["accession"]
                if accession in existing_ids:
                    print(f"  Skipping {accession} (already collected)")
                    continue

                sentences = collect_sentences_for_filing(ticker, cik, filing)
                remaining_for_company = MAX_SENTENCES_PER_COMPANY - company_counts[ticker]
                sentences = sentences[:remaining_for_company]
                sentence_counter = write_sentences(
                    writer,
                    sentences,
                    sentence_counter,
                    ticker,
                    cik,
                    filing,
                )
                company_counts[ticker] += len(sentences)
                csvfile.flush()
                existing_ids.add(accession)

                if sentence_counter >= TARGET_SENTENCES or company_counts[ticker] >= MAX_SENTENCES_PER_COMPANY:
                    break

            print(f"  Running total: {sentence_counter} sentences")

    print("\n" + "=" * 60)
    print(f"DONE. Collected {sentence_counter} sentences.")
    print(f"Saved to: {OUTPUT_FILE}")
    print("=" * 60)

    return read_saved_dataset(OUTPUT_FILE)

### Run Collection and Preview Results

The next cell runs the full collection. When it finishes, it prints the row count and shows the first 5 rows.

In [ ]:
raw_dataset = collect_dataset()

print(f"Rows collected: {len(raw_dataset):,}")
print(f"Columns: {list(raw_dataset.columns)}")

# The ticker cap is 80 companies * 200 sentences = about 16,000 possible rows.
if len(raw_dataset) < 15000:
    print("Warning: fewer than 15,000 rows were collected. Check the messages above for skipped filings or SEC request failures.")
else:
    print("Dataset size is in the expected ~16,000-row range.")

raw_dataset.head(5)

### Dataset Checks

These checks summarize the saved CSV and confirm that the dataset was written to the configured Google Drive path.

In [ ]:
saved_dataset = read_saved_dataset(OUTPUT_FILE)

print(f"Saved dataset path: {OUTPUT_FILE}")
print(f"Saved row count: {len(saved_dataset):,}")
print(f"Unique tickers: {saved_dataset['ticker'].nunique()}")
print(f"Unique filings: {saved_dataset['filing_id'].nunique()}")

# Show a small per-company count sample.
saved_dataset.groupby("ticker").size().sort_index().head(10).to_frame("sentence_count")

## Failures, Retries, and Limitations

- SEC requests use a descriptive `User-Agent` and a short pause after every request. This reduces the chance of rate-limit issues but does not eliminate network failures.
- The notebook duplicates the original script behavior: individual CIK lookup, filing retrieval, document download, and MD&A extraction failures are printed and skipped so the rest of the collection can continue.
- There is no automatic per-request retry loop in the original Step 4 code. If Colab disconnects or a request fails temporarily, rerun the collection cell. Because progress is saved to Google Drive after each filing, reruns resume from already-collected filing IDs.
- Before resuming, the notebook validates the existing `raw_dataset.csv`. If the file is malformed from a prior interrupted run, delete or rename it in Google Drive and rerun the notebook from the top.
- Google Drive must be mounted, and the `DRIVE_BASE` folder path must match the shared project folder exactly.
- MD&A extraction depends on text patterns such as `Item 7`, `Item 7A`, and `Item 8`. Some companies format filings differently, so a filing may be skipped if the MD&A section cannot be found reliably.
- The dataset size is approximate. With 80 tickers and a 200-sentence cap per company, the practical cap is about 16,000 rows, and the final count can vary if SEC content changes or some filings fail.